In [1]:
! pip install transformers[torch] datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.5/510.5 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 297.4/297.4 kB 9.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 7.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 8.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 8.0 MB/s eta 0:00:00
  Using cached nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (23.7 MB)
  Using cached nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (823 kB)
  Using cached nvidia_cuda_cupti_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (14.1 MB)
  Using cached nvidia_cudnn_cu12-8.9.2.26-py3-none-manylinux1_x86_64.whl (731.7 MB)
  Using cached nvidia_cublas_cu12-12.1.3.1-py3-none-manylinux1_x86_64.whl (410.6 MB)
  Using cached nvidia_cufft_cu12-11.0.2.54-py3-none-manylinux1_x86_64.whl (121.6 MB)
  Using cached nvidia_curand_cu12-10.3.2.106-py3-none-manylinux1_x86_64

# Causal language modeling

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Reading Mongolian inflection dataset

In [3]:
import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/unimelb/text_generation/data/khk.noun.tsv',sep='\t')

In [4]:
df.head()

,prompt,answer
0,багтраагаа,багтраа <voc>
1,дэнсээс,дэнс <abl>
2,тархалтыг,тархалт <acc>
3,дуулийг,дууль <acc>
4,гуалингаар,гуалин <ins>


In [5]:
import datasets
from datasets import Dataset, DatasetDict

df['prompt'] = '<s> bb: '+ df['prompt']
df['answer'] = df['answer']+'</s>'

infl_dataset = Dataset.from_pandas(df)

In [6]:
df.head()

,prompt,answer
0,<s> bb: багтраагаа,багтраа <voc></s>
1,<s> bb: дэнсээс,дэнс <abl></s>
2,<s> bb: тархалтыг,тархалт <acc></s>
3,<s> bb: дуулийг,дууль <acc></s>
4,<s> bb: гуалингаар,гуалин <ins></s>


Split the dataset's `train_asks` split into a train and test set with the [train_test_split](https://huggingface.co/docs/datasets/main/en/package_reference/main_classes#datasets.Dataset.train_test_split) method:

In [7]:
ds_train_devtest = infl_dataset.train_test_split(test_size=0.025, seed = 42)
ds_devtest = ds_train_devtest['test'].train_test_split(test_size=0.5, seed = 42)

ds_splits = DatasetDict({
    'train': ds_train_devtest['train'],
    'valid': ds_devtest['train'],
    'test': ds_devtest['test']
})


In [8]:
ds_splits

DatasetDict({
    train: Dataset({
        features: ['prompt', 'answer'],
        num_rows: 14036
    })
    valid: Dataset({
        features: ['prompt', 'answer'],
        num_rows: 180
    })
    test: Dataset({
        features: ['prompt', 'answer'],
        num_rows: 180
    })
})

Then take a look at an example:

In [9]:
def concatenate_columns(example):
    example["prompt"] = example["prompt"] + " " + example["answer"]
    return example

ds_splits["train"] = ds_splits["train"].map(concatenate_columns)
ds_splits["valid"] = ds_splits["valid"].map(concatenate_columns)

Map:   0%|          | 0/14036 [00:00<?, ? examples/s]

Map:   0%|          | 0/180 [00:00<?, ? examples/s]

In [10]:
ds_splits["train"][0:10]

{'prompt': ['<s> bb: хавчаарт хавчаар <dat></s>',
  '<s> bb: санхүүд санхүү <dat></s>',
  '<s> bb: зохицлын зохицол <gen></s>',
  '<s> bb: сейфээ сейф <voc></s>',
  '<s> bb: уулаа уул <voc></s>',
  '<s> bb: эхнэртэй эхнэр <com></s>',
  '<s> bb: ордноос ордон <abl></s>',
  '<s> bb: оноонд оноо <dat></s>',
  '<s> bb: нөөцийг нөөц <acc></s>',
  '<s> bb: цогцсыг цогцос <acc></s>'],
 'answer': ['хавчаар <dat></s>',
  'санхүү <dat></s>',
  'зохицол <gen></s>',
  'сейф <voc></s>',
  'уул <voc></s>',
  'эхнэр <com></s>',
  'ордон <abl></s>',
  'оноо <dat></s>',
  'нөөц <acc></s>',
  'цогцос <acc></s>']}

While this may look like a lot, you're only really interested in the `text` field. What's cool about language modeling
tasks is you don't need labels (also known as an unsupervised task) because the next word *is* the label.

## Preprocess

In [11]:
#@title
from IPython.display import HTML

HTML('<iframe width="560" height="315" src="https://www.youtube.com/embed/ma1TrR7gE7I?rel=0&amp;controls=0&amp;showinfo=0" frameborder="0" allowfullscreen></iframe>')

/usr/local/lib/python3.10/dist-packages/IPython/core/display.py:724: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


The next step is to load a DistilGPT2 tokenizer to process the `text` subfield:

In [12]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bayartsogt/mongolian-gpt2")

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:88: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/207 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.33M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.10M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/24.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/90.0 [00:00<?, ?B/s]

You'll notice from the example above, the `text` field is actually nested inside `answers`. This means you'll need to
extract the `text` subfield from its nested structure with the [`flatten`](https://huggingface.co/docs/datasets/process.html#flatten) method:

In [13]:
ds_splits = ds_splits.flatten()
ds_splits["train"][0]

{'prompt': '<s> bb: хавчаарт хавчаар <dat></s>', 'answer': 'хавчаар <dat></s>'}

Each subfield is now a separate column as indicated by the `answers` prefix, and the `text` field is a list now. Instead
of tokenizing each sentence separately, convert the list to a string so you can jointly tokenize them.

Here is a first preprocessing function to join the list of strings for each example and tokenize the result:

In [14]:
def preprocess_function(examples):
    return tokenizer(examples["prompt"])

To apply this preprocessing function over the entire dataset, use the 🤗 Datasets [map](https://huggingface.co/docs/datasets/main/en/package_reference/main_classes#datasets.Dataset.map) method. You can speed up the `map` function by setting `batched=True` to process multiple elements of the dataset at once, and increasing the number of processes with `num_proc`. Remove any columns you don't need:

In [15]:
tokenized_ds = ds_splits.map(
    preprocess_function,
    batched=True,
    num_proc=4,
    remove_columns=ds_splits["train"].column_names,
)

/usr/local/lib/python3.10/dist-packages/multiprocess/popen_fork.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = os.fork()


Map (num_proc=4):   0%|          | 0/14036 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/180 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/180 [00:00<?, ? examples/s]

This dataset contains the token sequences, but some of these are longer than the maximum input length for the model.

You can now use a second preprocessing function to
- concatenate all the sequences
- split the concatenated sequences into shorter chunks defined by `block_size`, which should be both shorter than the maximum input length and short enough for your GPU RAM.

In [16]:
block_size = 64


def group_texts(examples):
    # Concatenate all texts.
    concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated_examples[list(examples.keys())[0]])
    # We drop the small remainder, we could add padding if the model supported it instead of this drop, you can
    # customize this part to your needs.
    if total_length >= block_size:
        total_length = (total_length // block_size) * block_size
    # Split by chunks of block_size.
    result = {
        k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated_examples.items()
    }
    result["labels"] = result["input_ids"].copy()
    return result

Apply the `group_texts` function over the entire dataset:

In [17]:
lm_dataset = tokenized_ds.map(group_texts, batched=True, num_proc=4)

Map (num_proc=4):   0%|          | 0/14036 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/180 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/180 [00:00<?, ? examples/s]

Now create a batch of examples using [DataCollatorForLanguageModeling](https://huggingface.co/docs/transformers/main/en/main_classes/data_collator#transformers.DataCollatorForLanguageModeling). It's more efficient to *dynamically pad* the
sentences to the longest length in a batch during collation, instead of padding the whole dataset to the maximum length.

Use the end-of-sequence token as the padding token and set `mlm=False`. This will use the inputs as labels shifted to the right by one element:

In [18]:
from transformers import DataCollatorForLanguageModeling

tokenizer.pad_token = tokenizer.eos_token
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

## Train

<Tip>

If you aren't familiar with finetuning a model with the [Trainer](https://huggingface.co/docs/transformers/main/en/main_classes/trainer#transformers.Trainer), take a look at the [basic tutorial](https://huggingface.co/docs/transformers/main/en/tasks/../training#train-with-pytorch-trainer)!

</Tip>

You're ready to start training your model now! Load DistilGPT2 with [AutoModelForCausalLM](https://huggingface.co/docs/transformers/main/en/model_doc/auto#transformers.AutoModelForCausalLM):

In [19]:
from transformers import AutoModelForCausalLM, TrainingArguments, Trainer

model = AutoModelForCausalLM.from_pretrained("bayartsogt/mongolian-gpt2")

config.json:   0%|          | 0.00/864 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/510M [00:00<?, ?B/s]

At this point, only three steps remain:

1. Define your training hyperparameters in [TrainingArguments](https://huggingface.co/docs/transformers/main/en/main_classes/trainer#transformers.TrainingArguments). The only required parameter is `output_dir` which specifies where to save your model. You'll push this model to the Hub by setting `push_to_hub=True` (you need to be signed in to Hugging Face to upload your model).
2. Pass the training arguments to [Trainer](https://huggingface.co/docs/transformers/main/en/main_classes/trainer#transformers.Trainer) along with the model, datasets, and data collator.
3. Call [train()](https://huggingface.co/docs/transformers/main/en/main_classes/trainer#transformers.Trainer.train) to finetune your model.

In [20]:
training_args = TrainingArguments(
    output_dir="my_awesome_gpt-inflection-model",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    weight_decay=0.01,
    push_to_hub=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=lm_dataset["train"],
    eval_dataset=lm_dataset["test"],
    data_collator=data_collator,
)

trainer.train()

/usr/local/lib/python3.10/dist-packages/accelerate/accelerator.py:436: FutureWarning: Passing the following arguments to `Accelerator` is deprecated and will be removed in version 1.0 of Accelerate: dict_keys(['dispatch_batches', 'split_batches', 'even_batches', 'use_seedable_sampler']). Please pass an `accelerate.DataLoaderConfiguration` instead: 
dataloader_config = DataLoaderConfiguration(dispatch_batches=None, split_batches=False, even_batches=True, use_seedable_sampler=True)
  warnings.warn(


Epoch,Training Loss,Validation Loss
1,No log,5.545276
2,2.613700,5.652215
3,2.613700,5.753122


TrainOutput(global_step=969, training_loss=1.9840991135352166, metrics={'train_runtime': 186.1427, 'train_samples_per_second': 41.613, 'train_steps_per_second': 5.206, 'total_flos': 252996009984000.0, 'train_loss': 1.9840991135352166, 'epoch': 3.0})

Once training is completed, use the [evaluate()](https://huggingface.co/docs/transformers/main/en/main_classes/trainer#transformers.Trainer.evaluate) method to evaluate your model and get its perplexity:

In [28]:
import math

eval_results = trainer.evaluate()
print(f"Perplexity: {math.exp(eval_results['eval_loss']):.2f}")

RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cpu and cuda:0! (when checking argument for argument index in method wrapper_CUDA__index_select)

<Tip>

For a more in-depth example of how to finetune a model for causal language modeling, take a look at the corresponding
[PyTorch notebook](https://colab.research.google.com/github/huggingface/notebooks/blob/main/examples/language_modeling.ipynb)
or [TensorFlow notebook](https://colab.research.google.com/github/huggingface/notebooks/blob/main/examples/language_modeling-tf.ipynb).

</Tip>

## Inference

Great, now that you've finetuned a model, you can use it for inference!

Come up with a prompt you'd like to generate text from:

In [32]:
prompt = "<s> bb: сургуулийн"

The simplest way to try out your finetuned model for inference is to use it in a [pipeline()](https://huggingface.co/docs/transformers/main/en/main_classes/pipelines#transformers.pipeline). Instantiate a `pipeline` for text generation with your model, and pass your text to it:

In [30]:
from transformers import pipeline

generator = pipeline("text-generation", model=model, tokenizer = tokenizer, num_beams=5)

In [33]:
ans = generator(prompt, max_length = 15)

print(str(ans[0]['generated_text']))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


<s> bb: сургуулийн сургууль <gen> bb:


In [25]:
print(str(ans))

[{'generated_text': '<s> bb: сурагчаас сурагч <abl> bb'}]
